# Pareto Design Dashboard

## What you will learn
How final design decisions expose tradeoffs rather than producing one universally best stellarator.

## Codes used
Pure Python cached metrics table.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
`11_pareto_front.png` and `11_weighted_selection.png`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Keep RUN_MODE='cached' first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")

In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:
import importlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sos2026.paths import PROJECT_ROOT, DATA_DIR, CACHE_DIR, FIGURE_DIR, MOVIE_DIR, ensure_directories
ensure_directories()
print("Figures:", FIGURE_DIR)
print("Cached data:", CACHE_DIR)

## 1. Learning frame

This notebook is a deliberately small project: define one metric, produce one plot, expose one failure mode, and identify where a real code would enter.

In [ ]:
from sos2026.pareto import design_table, nondominated, weighted_selection
from sos2026.plotting import savefig, caption

## 2. Load or generate the teaching data

Cached mode uses small arrays so the conceptual workflow is always available.

In [ ]:
df = design_table()
df

## 3. Make the primary plot

Every plot has a one-sentence caption because students should know how to read it without guessing.

In [ ]:
keep = nondominated(df, ["epsilon_eff", "coil_length", "turbulence_proxy"])
fig, ax = plt.subplots(figsize=(6.4, 4.0))
sc = ax.scatter(df["coil_length"], df["epsilon_eff"], c=df["turbulence_proxy"], s=95, cmap="plasma", edgecolor="black", linewidth=0.5)
ax.scatter(df.loc[keep, "coil_length"], df.loc[keep, "epsilon_eff"], facecolors="none", edgecolors="lime", s=190, linewidths=2, label="nondominated")
for _, row in df.iterrows():
    ax.text(row["coil_length"] + 0.01, row["epsilon_eff"], row["design"], fontsize=7)
ax.set_xlabel("coil length proxy")
ax.set_ylabel("epsilon_eff proxy")
ax.set_title("Pareto front: transport vs coils")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
caption(ax, "Nondominated points expose tradeoffs that a single weighted objective can hide.")
savefig(fig, "11_pareto_front.png")
plt.show()

## 4. Probe the metric

A metric becomes useful for optimization only when we understand how it changes across design choices.

In [ ]:
weights = {"epsilon_eff": 0.35, "coil_length": 0.25, "turbulence_proxy": 0.25, "profile_error": 0.15}
ranked = weighted_selection(df, weights)
fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.bar(ranked["design"], ranked["weighted_score"], color="#0f766e")
ax.set_ylabel("weighted score, lower is better")
ax.set_title("Weighted selection dashboard")
ax.tick_params(axis="x", rotation=30)
caption(ax, "Changing weights is a design decision, not a physics result.")
savefig(fig, "11_weighted_selection.png")
plt.show()
ranked[["design", "weighted_score"]].head()

## 5. Interpret the design consequence

The table below translates the plot into an optimization decision.

In [ ]:
value_choices = pd.DataFrame({
    "decision": ["transport priority", "coil simplicity", "validation maturity", "profile confidence"],
    "effect": ["favors low epsilon/turbulence", "favors shorter/smoother coils", "penalizes unvalidated proxies", "favors profile-safe points"],
})
value_choices

## 6. Failure mode

The cached plot is useful only if we say what it does not prove.

In [ ]:
failure_mode = pd.DataFrame({
    "cached_mode_proves": ["workflow shape", "plot grammar", "where the metric enters"],
    "cached_mode_does_not_prove": ["validated physics", "final design ranking", "runtime scalability"],
})
failure_mode

## 7. Research-mode hook

Run this cell only after timing the package on the lecture machine.

In [ ]:
if RUN_MODE == "research":
    print("Research path: replace any cached metric column with a real output and recompute the front.")
else:
    print("Cached mode: research package path skipped intentionally.")

## 8. Mini project handoff

Use this notebook during the lecture as the computational project slide points to: change one parameter, regenerate one plot, and explain one design tradeoff.

In [ ]:
project_steps = pd.DataFrame({
    "step": [1, 2, 3, 4],
    "action": ["identify metric", "change one input", "regenerate plot", "state failure mode"],
})
project_steps

## Try this
Change one scalar or one row in the cached data and regenerate the primary plot.

## Expected qualitative answer
The plot should move in a physically interpretable direction, but the cached result remains an educational proxy.

## Research extension
Replace the cached data source with the corresponding real package output after timing and API verification.